# M3 phase B: Gengram (Megatron fork c7053faf) trained 200 steps from .bin/.idx and from panvram archives
Runtime: GPU (run on L4 2026-10-25). The fork patch (local branch panvram-m3, 4ee6879) is embedded below as base64; nothing is fetched from an unpublished repository.
Result of the run: bin 383.0 ms/iteration, 85 556 tokens/s, GPU util median 92 %; panvram 384.2 ms, 85 278 tokens/s, 91 %.

In [ ]:
#!/usr/bin/env python3
"""#M3 phase B - Gengram (Megatron fork c7053faf) trained 200 steps from two data paths on one GPU:
.bin/.idx (torch DataLoader, the fork as published) and panvram:<cohort> (patch 4ee6879: PanvramIndexedDataset +
device batch iterator). Same model, same seed, same GPTDataset indices -> same sample IDs. Printed: iteration time,
tokens/s, GPU utilisation (nvidia-smi every 200 ms), lm loss per 10 steps for both runs.
The .bin is written here from the same archive through Megatron's IndexedDatasetBuilder (one document per contig,
end-of-document id appended), then checked field by field against the panvram path before training.
"""
import base64, glob, json, os, re, subprocess, sys, time
from pathlib import Path

T0 = time.time()
def sh(cmd, cwd=None, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if check and r.returncode:
        print(f"$ {cmd}\n{r.stdout[-3000:]}\n{r.stderr[-3000:]}", flush=True); raise SystemExit(f"failed: {cmd}")
    return r.stdout
def stamp(msg): print(f"[{time.time()-T0:7.1f} s] {msg}", flush=True)

print(sh("nvidia-smi --query-gpu=name,memory.total,driver_version,clocks.max.sm --format=csv,noheader").strip(), flush=True)
sh("apt-get -qq install -y libzstd-dev")
sh("git clone -q https://github.com/yasha1971-coder/panvram.git /content/pvrepo || true")
sh("cd /content/pvrepo && pip install -q -e .")
stamp("panvram built")
sh("git clone -q https://github.com/BGI-HangzhouAI/Gengram.git /content/gg && cd /content/gg && git checkout -q c7053faf087acf05bf3edb8ebe2a22c84620ed9d")
Path("/content/m3.patch").write_bytes(base64.b64decode('...'))
sh("cd /content/gg && git apply /content/m3.patch && git log --oneline -1 && git status --short")
PY = sys.executable
sh(f"{PY} -m pip install -q regex einops pybind11")
sh(f"cd /content/gg/megatron/core/datasets && g++ -O3 -Wall -shared -std=c++11 -fPIC $({PY} -m pybind11 --includes) helpers.cpp "
   f"-o helpers_cpp$({PY} -c 'import sysconfig; print(sysconfig.get_config_var(\"EXT_SUFFIX\"))')")
sh(f"cd /content/gg && {PY} -c 'from megatron.core.datasets import helpers_cpp; print(helpers_cpp.__file__)'")
stamp("fork at c7053fa + panvram patch")
os.chdir("/tmp")
sys.path.insert(0, "/content/pvrepo"); sys.path.insert(0, "/content/gg")
sys.path.insert(0, "/content/pvrepo/logs/loader-2026-10-22/scripts")
from colab_prepare import prepare_cohort, _sha256
REF = ("https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/009/914/755/"
       "GCA_009914755.4_T2T-CHM13v2.0/GCA_009914755.4_T2T-CHM13v2.0_genomic.fna.gz")
sh("mkdir -p /content/cohort")
sh(f'curl -fsSL -o /content/t2t.fna.gz "{REF}"')
sh("gunzip -f /content/t2t.fna.gz && mv /content/t2t.fna /content/cohort/reference.fa")
assert _sha256("/content/cohort/reference.fa") == "51ab7d3c2a375ecff53ee2a1be03e58d57cc1601c948a860c03a78d724f26e4c"
prepare_cohort(5, dest="/content/cohort", work="/content/dl")
ASM = sorted(Path(p).name.split(".q16k")[0] for p in glob.glob("/content/cohort/*.q16k.rr3"))[0]
stamp(f"cohort ready; training assembly {ASM}")

# ---- .bin from the same archive, then a field check on the GPU path
import importlib; importlib.invalidate_caches()
import numpy as np, torch
os.environ["PANVRAM_ASSEMBLIES"] = ASM
os.environ["PANVRAM_DEVICE"] = "cuda"
from megatron.training.datasets.panvram_dataset import PanvramIndexedDataset
from megatron.core.datasets.indexed_dataset import IndexedDatasetBuilder, IndexedDataset
low = PanvramIndexedDataset.from_data_path("panvram:/content/cohort")
PFX = "/content/bin/" + ASM + "_text_document"
os.makedirs("/content/bin", exist_ok=True)
b = IndexedDatasetBuilder(PFX + ".bin", dtype=np.uint16)
for d in range(len(low)):
    x = low.get(d)
    b.add_document(x, [x.size])
b.finalize(PFX + ".idx")
ib = IndexedDataset(PFX)
assert np.array_equal(ib.sequence_lengths.astype(np.int64), low.sequence_lengths.astype(np.int64))
stamp(f".bin written: {os.path.getsize(PFX + '.bin')} B, {len(low)} documents, archive {os.path.getsize(glob.glob('/content/cohort/' + ASM + '*.rr3')[0])} B")
del low, ib; torch.cuda.empty_cache()

SEQ, MBS, GBS, ITERS = int(os.environ.get("SEQ", "2048")), 16, 16, 200
COMMON = f"""--num-layers 4 --hidden-size 256 --num-attention-heads 4 --seq-length {SEQ} --max-position-embeddings {SEQ}
 --transformer-impl local --normalization RMSNorm --position-embedding-type rope
 --gengram-enabled --gengram-layer-ids 2 --gengram-ngram-sizes 2 3 4 5 6 --gengram-embed-dim-per-ngram 64
 --gengram-token-ids 8 5 6 7 9 --gengram-use-short-conv --gengram-short-conv-kernel-size 4 --gengram-window-size 12
 --micro-batch-size {MBS} --global-batch-size {GBS} --train-iters {ITERS} --lr 3e-4 --min-lr 3e-5 --lr-decay-style cosine
 --lr-warmup-iters 20 --weight-decay 0.1 --clip-grad 1.0 --bf16 --seed 1234 --init-method-std 0.02
 --tokenizer-type NullTokenizer --vocab-size 12 --split 100,0,0 --eval-iters 0 --eval-interval 1000000
 --log-interval 10 --log-throughput --no-gradient-accumulation-fusion --tensor-model-parallel-size 1
 --pipeline-model-parallel-size 1 --reset-position-ids --eod-mask-loss --num-workers 2
 --no-create-attention-mask-in-dataloader --no-rope-fusion --no-persist-layer-norm --no-masked-softmax-fusion"""
def train(tag, data_path):
    log = f"/content/train_{tag}.log"
    smi = subprocess.Popen(f"nvidia-smi --query-gpu=timestamp,utilization.gpu,memory.used --format=csv,noheader -lms 200 > /content/smi_{tag}.csv", shell=True)
    env = dict(os.environ, CUDA_DEVICE_MAX_CONNECTIONS="1", PYTHONPATH="/content/gg:/content/pvrepo",
               TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD="1")
    t = time.time()
    r = subprocess.run(f"{sys.executable} -m torch.distributed.run --nproc_per_node 1 --master_port 6005 pretrain_gpt.py {' '.join(COMMON.split())} --data-path {data_path} > {log} 2>&1",
                       shell=True, cwd="/content/gg", env=env)
    wall = time.time() - t
    smi.terminate(); time.sleep(1)
    txt = open(log).read()
    if r.returncode:
        print(txt[-5000:], flush=True); raise SystemExit(f"training {tag} failed")
    its = [float(x) for x in re.findall(r"elapsed time per iteration \(ms\): ([0-9.]+)", txt)]
    loss = [float(x) for x in re.findall(r"lm loss: ([0-9.E+-]+)", txt)]
    util = []
    for l in open(f"/content/smi_{tag}.csv"):
        p = l.split(",")
        if len(p) >= 2 and p[1].strip().endswith("%"): util.append(int(p[1].strip()[:-1]))
    steady = its[2:] if len(its) > 3 else its
    ms = float(np.median(steady)) if steady else float("nan")
    res = {"tag": tag, "wall_s": wall, "iter_ms_per_log": its, "iter_ms_median_after_20": ms,
           "tokens_per_s": GBS * SEQ / (ms / 1e3), "lm_loss": loss,
           "gpu_util_median": float(np.median(util)) if util else None,
           "gpu_util_mean": float(np.mean(util)) if util else None, "util_samples": len(util)}
    stamp(f"{tag}: iteration {ms:.1f} ms median (after 20), {res['tokens_per_s']:.0f} tokens/s, GPU util median {res['gpu_util_median']} % mean {res['gpu_util_mean']:.1f} %, wall {wall:.0f} s")
    print(f"   lm loss every 10 steps: {loss}", flush=True)
    return res, txt

out = {"assembly": ASM, "seq": SEQ, "mbs": MBS, "gbs": GBS, "iters": ITERS}
out["bin"], tb = train("bin", PFX)
out["panvram"], tp = train("panvram", "panvram:/content/cohort")
out["loss_identical"] = out["bin"]["lm_loss"] == out["panvram"]["lm_loss"]
print("lm loss identical in every logged step:", out["loss_identical"], flush=True)
print("panvram index cache directories (PanvramIndexedDataset was built by the training run):",
      sorted(p.name for p in Path("/content/cohort").glob("panvram-*")), flush=True)
Path("/content/m3b.json").write_text(json.dumps(out, indent=1))
print(json.dumps({k: (v if not isinstance(v, dict) else {kk: vv for kk, vv in v.items() if kk not in ("iter_ms_per_log",)}) for k, v in out.items()}, indent=1), flush=True)
print("DONE", flush=True)
